# Tabela 8 — Logit Multinomial
## Forma de pagamento do exame preventivo (Papanicolau)
### PNS 2013 e 2019 (Pooled)

---

**Objetivo:** Estimar um modelo Logit Multinomial para analisar os determinantes da forma de pagamento do exame preventivo do colo do útero (Papanicolau) em mulheres brasileiras.

**Abordagem:** Utilização de abstração semântica (variáveis do `mapping.py`), sem referência direta a códigos físicos da PNS.

**Estrutura do Notebook:**
1. Importações
2. Carregamento dos dados
3. Construção da variável dependente (multinomial)
4. Construção das variáveis explicativas
5. Diagnóstico pré-modelo
6. Estimação do Logit Multinomial (ÚNICO MODELO)
7. Resultados: Tabela 8A (Coeficientes e RRR)
8. Resultados: Tabela 8B (Efeitos Marginais Médios)
9. Estatísticas globais do modelo
10. Exportação de resultados

## 1. IMPORTAÇÕES

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import sys

sys.path.append("..")
from service.pns_service import get_dataframe

# Configurações para melhor legibilidade
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
np.set_printoptions(suppress=True)

## 2. CARREGAMENTO DOS DADOS (ABSTRAÇÃO SEMÂNTICA)

Usamos **exclusivamente variáveis semânticas** definidas em `mapping.py`.
Nenhum código físico da PNS aparece neste notebook.

In [2]:
# Variáveis semânticas necessárias
variaveis_necessarias = [
    "idade",
    "vive_com_companheiro",
    "tem_filhos_nascidos_vivos",
    "trabalha",
    "escolaridade_nivel",
    "renda_domiciliar_pc",
    "situacao_censitaria",
    "preventivo_quando",
    "preventivo_sus",
    "preventivo_plano",
    "preventivo_pago",
]

# Carregamento dos dados
print("\n" + "="*70)
print("CARREGAMENTO DOS DADOS")
print("="*70)

df_bruto = get_dataframe(
    variables=variaveis_necessarias,
    sources=["2013", "2019"],
    filters={
        "sexo": "2",                              # Apenas mulheres
        "idade": {"operador": ">=", "valor": 25}  # Idade >= 25 anos
    }
)

print(f"\n✓ Amostra bruta: {len(df_bruto):,} observações")
print(f"✓ Período: 2013 e 2019 (pooled)")
print(f"\nDimensões: {df_bruto.shape}")
print(f"\nPrimeiras 5 linhas:")
print(df_bruto.head())


CARREGAMENTO DOS DADOS

✓ Amostra bruta: 158,912 observações
✓ Período: 2013 e 2019 (pooled)

Dimensões: (158912, 15)

Primeiras 5 linhas:
  preventivo_pago preventivo_quando  tem_filhos_nascidos_vivos  \
0             não                 2                        3.0   
1   não informado                 5                        1.0   
2             não                 3                        3.0   
3             não                 2                        2.0   
4   não informado              None                        NaN   

  situacao_censitaria   id_upa id_morador  renda_domiciliar_pc origem  \
0                   2  1100001          1                878.0   2013   
1                   2  1100001          1                300.0   2013   
2                   2  1100001          2                174.0   2013   
3                   2  1100001          1                650.0   2013   
4                   2  1100001          2                700.0   2013   

  preventivo_sus prevent

## 3. CONSTRUÇÃO DA VARIÁVEL DEPENDENTE (MULTINOMIAL)

Codifica mutuamente exclusivamente a forma de realização do exame preventivo:
- **y = 0:** Realizado pelo SUS (referência)
- **y = 1:** Coberto por plano de saúde privado
- **y = 2:** Pago diretamente pela mulher

In [3]:
def build_multinomial_outcome_variable(df_entrada):
    """Constrói variável dependente multinomial com validação rigorosa."""
    
    df_limpo = df_entrada.copy()
    n_inicio = len(df_limpo)
    
    print("\n" + "="*70)
    print("CONSTRUÇÃO DA VARIÁVEL DEPENDENTE")
    print("="*70)
    
    # Passo 1: Excluir mulheres que nunca fizeram o exame
    print("\n[Passo 1] Filtrar por realização do exame...")
    df_limpo = df_limpo[df_limpo["preventivo_quando"] != "5"].copy()
    n_apos_filtro = len(df_limpo)
    print(f"  → Removidas (nunca fizeram): {n_inicio - n_apos_filtro}")
    print(f"  → Amostra restante: {n_apos_filtro:,}")
    
    # Passo 2: Standardizar valores binárias
    print("\n[Passo 2] Standardizar valores binárias...")
    
    for var in ["preventivo_sus", "preventivo_plano", "preventivo_pago"]:
        df_limpo[var] = (
            df_limpo[var]
            .astype(str).str.strip().str.lower()
            .map({"1": 1, "2": 0, "sim": 1, "não": 0, "nao": 0})
        )
    
    print("  ✓ Variáveis binárias standardizadas (0/1)")
    
    # Passo 3: Codificar categorias mutuamente exclusivas
    print("\n[Passo 3] Codificar forma de pagamento (categorias exclusivas)...")
    
    df_limpo["y_forma_pagamento"] = np.nan
    
    # Categoria 0: SUS
    idx_sus = (df_limpo["preventivo_sus"] == 1) & \
              (df_limpo["preventivo_plano"] == 0) & \
              (df_limpo["preventivo_pago"] == 0)
    df_limpo.loc[idx_sus, "y_forma_pagamento"] = 0
    
    # Categoria 1: Plano
    idx_plano = (df_limpo["preventivo_plano"] == 1) & \
                (df_limpo["preventivo_sus"] == 0) & \
                (df_limpo["preventivo_pago"] == 0)
    df_limpo.loc[idx_plano, "y_forma_pagamento"] = 1
    
    # Categoria 2: Pagou
    idx_pago = (df_limpo["preventivo_pago"] == 1) & \
               (df_limpo["preventivo_sus"] == 0) & \
               (df_limpo["preventivo_plano"] == 0)
    df_limpo.loc[idx_pago, "y_forma_pagamento"] = 2
    
    print(f"  → SUS: {idx_sus.sum():,}")
    print(f"  → Plano: {idx_plano.sum():,}")
    print(f"  → Pagou: {idx_pago.sum():,}")
    
    # Passo 4: Remover observações inconsistentes
    print("\n[Passo 4] Remover observações inconsistentes...")
    n_apos_cod = len(df_limpo)
    df_limpo = df_limpo.dropna(subset=["y_forma_pagamento"])
    n_final = len(df_limpo)
    print(f"  → Removidas (inconsistências): {n_apos_cod - n_final}")
    print(f"  → Amostra final: {n_final:,}")
    
    df_limpo["y_forma_pagamento"] = df_limpo["y_forma_pagamento"].astype(int)
    
    print("\n✓ Variável dependente construída com sucesso!")
    print("\nDistribuição:")
    for cat, label in enumerate(["SUS", "Plano", "Pagou"]):
        freq = (df_limpo["y_forma_pagamento"] == cat).sum()
        pct = 100 * freq / n_final
        print(f"  {cat} ({label}): {freq:,} ({pct:.1f}%)")
    
    return df_limpo, df_limpo["y_forma_pagamento"]


df_limpo, y_forma_pagamento = build_multinomial_outcome_variable(df_bruto)


CONSTRUÇÃO DA VARIÁVEL DEPENDENTE

[Passo 1] Filtrar por realização do exame...
  → Removidas (nunca fizeram): 6799
  → Amostra restante: 152,113

[Passo 2] Standardizar valores binárias...
  ✓ Variáveis binárias standardizadas (0/1)

[Passo 3] Codificar forma de pagamento (categorias exclusivas)...
  → SUS: 14,431
  → Plano: 5,961
  → Pagou: 3,320

[Passo 4] Remover observações inconsistentes...
  → Removidas (inconsistências): 128401
  → Amostra final: 23,712

✓ Variável dependente construída com sucesso!

Distribuição:
  0 (SUS): 14,431 (60.9%)
  1 (Plano): 5,961 (25.1%)
  2 (Pagou): 3,320 (14.0%)


## 4. CONSTRUÇÃO DAS VARIÁVEIS EXPLICATIVAS

Construção robusta de variáveis explicativas com transparência total.

In [4]:
def build_explanatory_variables(df_entrada):
    """Constrói matriz de variáveis explicativas robustamente."""
    
    df_vars = df_entrada.copy()
    
    print("\n" + "="*70)
    print("CONSTRUÇÃO DE VARIÁVEIS EXPLICATIVAS")
    print("="*70)
    
    # 1. FAIXAS ETÁRIAS
    print("\n[1] Faixas etárias...")
    df_vars["idade"] = pd.to_numeric(df_vars["idade"], errors="coerce")
    
    df_vars["idade_25_34"] = ((df_vars["idade"] >= 25) & (df_vars["idade"] <= 34)).astype(int)
    df_vars["idade_35_44"] = ((df_vars["idade"] >= 35) & (df_vars["idade"] <= 44)).astype(int)
    df_vars["idade_45_54"] = ((df_vars["idade"] >= 45) & (df_vars["idade"] <= 54)).astype(int)
    df_vars["idade_65_mais"] = (df_vars["idade"] >= 65).astype(int)
    # Ref: 55-64 anos (omitida)
    
    print(f"  25-34: {df_vars['idade_25_34'].sum()}; 35-44: {df_vars['idade_35_44'].sum()}")
    print(f"  45-54: {df_vars['idade_45_54'].sum()}; 65+: {df_vars['idade_65_mais'].sum()}")
    
    # 2. ESTRUTURA FAMILIAR
    print("\n[2] Estrutura familiar...")
    
    df_vars["vive_com_companheiro"] = (
        df_vars["vive_com_companheiro"].astype(str).str.strip().str.lower()
        .map({"1": 1, "2": 0, "sim": 1, "não": 0, "nao": 0})
    )
    
    df_vars["tem_filhos_nascidos_vivos"] = pd.to_numeric(
        df_vars["tem_filhos_nascidos_vivos"], errors="coerce"
    )
    df_vars["tem_filhos"] = (df_vars["tem_filhos_nascidos_vivos"] > 0).astype(int)
    
    print(f"  Vive com companheiro: {df_vars['vive_com_companheiro'].sum()}")
    print(f"  Tem filhos: {df_vars['tem_filhos'].sum()}")
    
    # 3. TRABALHO
    print("\n[3] Inserção no mercado de trabalho...")
    
    df_vars["trabalha"] = (
        df_vars["trabalha"].astype(str).str.strip().str.lower()
        .map({"1": 1, "2": 0, "sim": 1, "não": 0, "nao": 0})
    )
    
    print(f"  Trabalha (ocupada): {df_vars['trabalha'].sum()}")
    
    # 4. ESCOLARIDADE
    print("\n[4] Escolaridade...")
    
    mapa_esc = {1: 0, 2: 4, 3: 8, 4: 11, 5: 15, 6: 18}
    df_vars["anos_estudo"] = pd.to_numeric(
        df_vars["escolaridade_nivel"], errors="coerce"
    ).map(mapa_esc)
    
    print(f"  Média: {df_vars['anos_estudo'].mean():.2f} anos")
    
    # 5. LOCALIZAÇÃO
    print("\n[5] Localização...")
    
    df_vars["urbano"] = (
        df_vars["situacao_censitaria"].astype(str).str.strip() == "1"
    ).astype(int)
    
    print(f"  Urbana: {df_vars['urbano'].sum()}")
    
    # 6. RENDA (DECIS)
    print("\n[6] Decis de renda...")
    
    df_vars["renda_domiciliar_pc"] = pd.to_numeric(
        df_vars["renda_domiciliar_pc"], errors="coerce"
    )
    
    # Detectar coluna de ano
    ano_col = None
    for col in ["origem", "ano", "year"]:
        if col in df_vars.columns:
            ano_col = col
            break
    
    if ano_col:
        df_vars["decil_renda"] = (
            df_vars.groupby(ano_col)["renda_domiciliar_pc"]
            .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates="drop") + 1)
        )
    else:
        df_vars["decil_renda"] = pd.qcut(
            df_vars["renda_domiciliar_pc"], 10, labels=False, duplicates="drop"
        ) + 1
    
    dummies_decis = pd.get_dummies(df_vars["decil_renda"], prefix="decil", drop_first=True)
    df_vars = pd.concat([df_vars, dummies_decis], axis=1)
    
    print(f"  Dummies criadas: {list(dummies_decis.columns)[:3]}...")
    
    # SELEÇÃO FINAL
    print("\n[7] Seleção final de variáveis...")
    
    variaveis_explicativas = [
        "idade_25_34", "idade_35_44", "idade_45_54", "idade_65_mais",
        "vive_com_companheiro", "tem_filhos", "trabalha",
        "anos_estudo", "urbano"
    ] + list(dummies_decis.columns)
    
    # LIMPEZA
    faltantes_antes = df_vars[variaveis_explicativas].isna().sum().sum()
    df_vars = df_vars.dropna(subset=variaveis_explicativas)
    faltantes_depois = df_vars[variaveis_explicativas].isna().sum().sum()
    
    print(f"  NaN removidos: {faltantes_antes}")
    print(f"  Observações retidas: {len(df_vars):,}")
    
    X_matriz = df_vars[variaveis_explicativas].astype(float)
    
    assert X_matriz.isna().sum().sum() == 0, "Erro: ainda há NaN em X"
    assert not any(X_matriz.dtypes == "object"), "Erro: X contém colunas object"
    
    print(f"\n✓ Variáveis explicativas construídas!")
    print(f"  Dimensão: {X_matriz.shape}")
    
    return df_vars, X_matriz, variaveis_explicativas


df, X_matriz_sem_constante, lista_vars_explicativas = build_explanatory_variables(df_limpo)


CONSTRUÇÃO DE VARIÁVEIS EXPLICATIVAS

[1] Faixas etárias...
  25-34: 6340; 35-44: 6006
  45-54: 4672; 65+: 3171

[2] Estrutura familiar...
  Vive com companheiro: 13730
  Tem filhos: 11853

[3] Inserção no mercado de trabalho...
  Trabalha (ocupada): 12321.0

[4] Escolaridade...
  Média: 8.55 anos

[5] Localização...
  Urbana: 20126

[6] Decis de renda...
  Dummies criadas: ['decil_2.0', 'decil_3.0', 'decil_4.0']...

[7] Seleção final de variáveis...
  NaN removidos: 14265
  Observações retidas: 10,270

✓ Variáveis explicativas construídas!
  Dimensão: (10270, 18)


## 5. DIAGNÓSTICO PRÉ-MODELO

In [5]:
# Alinhamento final de X e y
indices_comuns = X_matriz_sem_constante.index.intersection(y_forma_pagamento.index)

X_final = X_matriz_sem_constante.loc[indices_comuns].copy()
y_final = y_forma_pagamento.loc[indices_comuns].copy()

print(f"\nAlinhamento de índices:")
print(f"  X: {X_final.shape[0]:,} obs × {X_final.shape[1]} vars")
print(f"  y: {len(y_final):,} obs")
print(f"  ✓ Alinhados: {X_final.shape[0] == len(y_final)}")

# Diagnóstico
print("\n" + "="*70)
print("DIAGNÓSTICO PRÉ-MODELO")
print("="*70)

print(f"\n[✓] Dimensões:")
print(f"  Amostra final: {len(y_final):,} observações")
print(f"  Variáveis: {X_final.shape[1]}")

print(f"\n[✓] Valores faltantes:")
print(f"  NaN em X: {X_final.isna().sum().sum()}")
print(f"  NaN em y: {y_final.isna().sum()}")

print(f"\n[✓] Tipos de dados:")
print(f"  X: {X_final.dtypes.unique()}")
print(f"  y: {y_final.dtype}")

print(f"\n[✓] Distribuição de y:")
for cat, label in enumerate(["SUS", "Plano", "Pagou"]):
    freq = (y_final == cat).sum()
    pct = 100 * freq / len(y_final)
    print(f"  {cat} ({label}): {freq:,} ({pct:.1f}%)")

assert X_final.shape[0] == len(y_final), "Erro: X e y desalinhados"
assert X_final.isna().sum().sum() == 0, "Erro: NaN em X"
assert y_final.isna().sum() == 0, "Erro: NaN em y"
assert y_final.nunique() == 3, "Erro: y deve ter 3 categorias"

print("\n✓ DADOS PRONTOS PARA MODELAGEM!")


Alinhamento de índices:
  X: 10,270 obs × 18 vars
  y: 10,270 obs
  ✓ Alinhados: True

DIAGNÓSTICO PRÉ-MODELO

[✓] Dimensões:
  Amostra final: 10,270 observações
  Variáveis: 18

[✓] Valores faltantes:
  NaN em X: 0
  NaN em y: 0

[✓] Tipos de dados:
  X: [dtype('float64')]
  y: int64

[✓] Distribuição de y:
  0 (SUS): 6,855 (66.7%)
  1 (Plano): 1,879 (18.3%)
  2 (Pagou): 1,536 (15.0%)

✓ DADOS PRONTOS PARA MODELAGEM!


## 6. ESTIMAÇÃO DO LOGIT MULTINOMIAL

**Especificação:**
$$\Pr(Y_i = j | \mathbf{X}_i) = \frac{\exp(\mathbf{X}_i \boldsymbol{\beta}_j)}{\sum_{k=0}^{2} \exp(\mathbf{X}_i \boldsymbol{\beta}_k)}$$

onde:
- $Y_i \in \{0, 1, 2\}$ é a forma de pagamento
- $j \in \{1, 2\}$ são as categorias (Plano, Pagou)
- Categoria 0 (SUS) é referência com $\boldsymbol{\beta}_0 = 0$
- Método: Newton com máximo 100 iterações

In [9]:
print("\n" + "="*70)
print("ESTIMAÇÃO DO MODELO LOGIT MULTINOMIAL")
print("="*70)

# Preparar dados: adicionar constante a X_final
X_modelo = sm.add_constant(X_final, has_constant="add").astype(float)
y_modelo = y_final.copy()

print(f"\nEspecificação:")
print(f"  Modelo: Logit Multinomial")
print(f"  Variável dependente: Forma de pagamento (3 categorias)")
print(f"  Categoria de referência: 0 (SUS)")
print(f"  Método: Newton")
print(f"  Amostra: {len(y_modelo):,} observações")
print(f"  Variáveis: {X_modelo.shape[1]} (com constante)")

# Instanciar e estimar
modelo_estimado = sm.MNLogit(
    y_modelo,
    X_modelo
).fit(
    method="newton",
    maxiter=100,
    disp=False,
    cov_type="HC0"   # <<< AQUI
)


print(f"\n✓ ESTIMAÇÃO CONCLUÍDA COM SUCESSO!")

# Verificar convergência
if modelo_estimado.mle_retvals['converged']:
    print(f"✓ Convergência: SIM")
else:
    print(f"⚠ Convergência: NÃO (verifique especificação)")

iteracoes = modelo_estimado.mle_retvals['iterations']
print(f"✓ Iterações: {iteracoes}")

# Mapeamento de categorias
categorias_nomes = {0: "SUS", 1: "Plano de Saúde", 2: "Pagou"}

print(f"\n" + "="*70)
print(f"SUMÁRIO DO MODELO ESTIMADO")
print(f"="*70)
print(f"\n{modelo_estimado.summary()}")


ESTIMAÇÃO DO MODELO LOGIT MULTINOMIAL

Especificação:
  Modelo: Logit Multinomial
  Variável dependente: Forma de pagamento (3 categorias)
  Categoria de referência: 0 (SUS)
  Método: Newton
  Amostra: 10,270 observações
  Variáveis: 19 (com constante)

✓ ESTIMAÇÃO CONCLUÍDA COM SUCESSO!
✓ Convergência: SIM
✓ Iterações: 7

SUMÁRIO DO MODELO ESTIMADO

                          MNLogit Regression Results                          
Dep. Variable:      y_forma_pagamento   No. Observations:                10270
Model:                        MNLogit   Df Residuals:                    10232
Method:                           MLE   Df Model:                           36
Date:                Sun, 25 Jan 2026   Pseudo R-squ.:                  0.1274
Time:                        09:56:32   Log-Likelihood:                -7749.4
converged:                       True   LL-Null:                       -8881.1
Covariance Type:                  HC0   LLR p-value:                     0.000
 y_forma_pagam

## 7. TABELA 8A: COEFICIENTES E RAZÕES DE RISCO RELATIVO

**Nota:** Apenas categorias 1 (Plano de Saúde) e 2 (Pagou).
Categoria 0 (SUS) é referência e não apresenta coeficientes.

In [10]:
# Mapeamento correto: colunas do modelo → rótulos substantivos
mapa_categorias = {
    0: "SUS",
    1: "Plano",
    2: "Pagou"
}


In [12]:
# ============================================================
# TABELA 8A — COEFICIENTES E RAZÕES DE RISCO RELATIVO (RRR)
# ============================================================

# Função auxiliar para significância
def asteriscos_sig(p_val):
    if p_val < 0.01:
        return "***"
    elif p_val < 0.05:
        return "**"
    elif p_val < 0.10:
        return "*"
    return ""

# ------------------------------------------------------------
# Extrair componentes (modelo já estimado com covariância robusta)
# ------------------------------------------------------------
coefs = modelo_estimado.params
p_vals = modelo_estimado.pvalues
erros_padrao = modelo_estimado.bse
rrrs = np.exp(coefs)

print("\n" + "="*70)
print("TABELA 8A: COEFICIENTES E RAZÕES DE RISCO RELATIVO (RRR)")
print("="*70)
print("\nVariável dependente: Forma de pagamento do exame preventivo")
print("Categoria de referência: SUS (y=0)")
print("\n")

# ------------------------------------------------------------
# Construção da tabela
# ------------------------------------------------------------
tabela_8a_linhas = []

for cat in coefs.columns:
    cat_nome = mapa_categorias.get(cat, f"Categoria {cat}")
    
    print(f"{cat_nome}:")
    print("-" * 100)
    
    cat_linhas = []
    
    for var_nome in coefs.index:
        coef_val = coefs.loc[var_nome, cat]
        p_val = p_vals.loc[var_nome, cat]
        rrr_val = rrrs.loc[var_nome, cat]
        ep = erros_padrao.loc[var_nome, cat]
        sig = asteriscos_sig(p_val)
        
        cat_linhas.append({
            "Variável": var_nome,
            "Coeficiente": f"{coef_val:9.4f}{sig}",
            "Erro Padrão": f"{ep:9.4f}",
            "RRR": f"{rrr_val:8.4f}",
            "P-valor": f"{p_val:8.4f}"
        })
        
        tabela_8a_linhas.append({
            "Categoria": cat_nome,
            "Variável": var_nome,
            "Coeficiente": f"{coef_val:9.4f}{sig}",
            "E.P.": f"{ep:9.4f}",
            "RRR": f"{rrr_val:8.4f}",
            "P-valor": f"{p_val:8.4f}"
        })
    
    df_cat = pd.DataFrame(cat_linhas)
    print(df_cat.to_string(index=False))
    print()

# DataFrame final
tabela_8a = pd.DataFrame(tabela_8a_linhas)

print("\n" + "="*100)
print("Notas:")
print("  *** p < 0.01 (significância a 1%)")
print("  **  p < 0.05 (significância a 5%)")
print("  *   p < 0.10 (significância a 10%)")
print("  RRR = Razão de Risco Relativo = exp(Coeficiente)")
print("  Erros-padrão robustos de White (HC0)")



TABELA 8A: COEFICIENTES E RAZÕES DE RISCO RELATIVO (RRR)

Variável dependente: Forma de pagamento do exame preventivo
Categoria de referência: SUS (y=0)


SUS:
----------------------------------------------------------------------------------------------------
            Variável  Coeficiente Erro Padrão      RRR  P-valor
               const   -5.6214***      0.2871   0.0036   0.0000
         idade_25_34       0.0472      0.1207   1.0483   0.6957
         idade_35_44       0.1404      0.1225   1.1507   0.2518
         idade_45_54      -0.0366      0.1163   0.9640   0.7527
       idade_65_mais    0.5021***      0.1824   1.6521   0.0059
vive_com_companheiro    0.2517***      0.0625   1.2862   0.0001
          tem_filhos       0.1083      0.0747   1.1144   0.1471
            trabalha      -0.0592      0.1484   0.9426   0.6902
         anos_estudo    0.1236***      0.0066   1.1315   0.0000
              urbano    1.0896***      0.1354   2.9732   0.0000
           decil_2.0       0.3131 

## 8. TABELA 8B: EFEITOS MARGINAIS MÉDIOS (AME)

Os efeitos marginais médios representam a mudança na probabilidade predita de cada categoria resultante de um aumento unitário nas variáveis explicativas (continuas) ou mudança de 0 para 1 (variáveis binárias).

In [14]:
# ============================================================
# TABELA 8B — EFEITOS MARGINAIS MÉDIOS (AME)
# ============================================================

print("\n" + "="*70)
print("TABELA 8B: EFEITOS MARGINAIS MÉDIOS (AME)")
print("="*70)
print("\nCategoria de referência: SUS (y=0)")
print("AME calculados via statsmodels (derivada analítica)\n")

# Cálculo correto dos efeitos marginais médios
marginais = modelo_estimado.get_margeff(
    at="overall",
    method="dydx"
)

# Resumo em DataFrame
df_marginais = marginais.summary_frame()

# Resetar índice para facilitar manipulação
df_marginais = df_marginais.reset_index()

# Mapear categorias corretamente
mapa_cat = {
    "y_forma_pagamento=0": "SUS",
    "y_forma_pagamento=1": "Plano",
    "y_forma_pagamento=2": "Pagou"
}

df_marginais["Outcome"] = df_marginais["endog"].map(mapa_cat)

# Renomear colunas para padrão de artigo
df_marginais = df_marginais.rename(columns={
    "exog": "Variável",
    "dy/dx": "AME",
    "Std. Err.": "Erro Padrão",
    "Pr(>|z|)": "P-valor"
})

# Asteriscos de significância
def asteriscos(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

df_marginais["AME_fmt"] = (
    df_marginais["AME"]
    .round(4)
    .astype(str)
    + df_marginais["P-valor"].apply(asteriscos)
)

# Selecionar colunas finais
tabela_8b = df_marginais[
    ["Variável", "Outcome", "AME_fmt", "Erro Padrão", "P-valor"]
]

# Exibir por categoria
for cat in ["SUS", "Plano", "Pagou"]:
    print(f"\n{cat}:")
    print("-" * 90)
    df_cat = tabela_8b[tabela_8b["Outcome"] == cat].drop("Outcome", axis=1)
    print(df_cat.to_string(index=False))

print("\n" + "="*70)
print("Notas:")
print("  • AME = efeito marginal médio")
print("  • Valores representam variação na probabilidade")
print("  • Método: derivada analítica (statsmodels)")



TABELA 8B: EFEITOS MARGINAIS MÉDIOS (AME)

Categoria de referência: SUS (y=0)
AME calculados via statsmodels (derivada analítica)


SUS:
------------------------------------------------------------------------------------------
            Variável    AME_fmt  Erro Padrão      P-valor
         idade_25_34 -0.0792***     0.018581 2.000604e-05
         idade_35_44 -0.0781***     0.018882 3.516726e-05
         idade_45_54  -0.0394**     0.017785 2.686705e-02
       idade_65_mais   -0.0586*     0.030707 5.642522e-02
vive_com_companheiro -0.0315***     0.009005 4.722604e-04
          tem_filhos    -0.0094     0.011016 3.935156e-01
            trabalha     0.0075     0.019810 7.044778e-01
         anos_estudo -0.0157***     0.000837 1.608007e-78
              urbano -0.0894***     0.015667 1.140761e-08
           decil_2.0    -0.0345     0.027623 2.119549e-01
           decil_3.0  -0.0662**     0.026049 1.106448e-02
           decil_4.0 -0.1456***     0.024557 3.037610e-09
           decil_

## 9. ESTATÍSTICAS GLOBAIS DO MODELO

Avaliação da qualidade geral do ajuste e capacidade explicativa.

In [15]:
# ============================================================
# ESTATÍSTICAS GLOBAIS DO MODELO
# ============================================================
# Qualidade geral do ajuste e capacidade explicativa

print("\n" + "="*70)
print("ESTATÍSTICAS GLOBAIS DO MODELO")
print("="*70)

nobs_modelo = int(modelo_estimado.nobs)
ll_modelo = modelo_estimado.llf
lr_chi2_valor = modelo_estimado.llr
p_chi2_valor = modelo_estimado.llr_pvalue
pseudo_r2_mcfadden = modelo_estimado.prsquared

print(f"\nObservações: {nobs_modelo:,}")
print(f"  → Mulheres entrevistadas com dados completos\n")

print(f"Log-Likelihood: {ll_modelo:12.4f}")
print(f"  → Valor da função de verossimilhança no ponto ótimo")
print(f"  → Quanto maior (menos negativo), melhor o ajuste\n")

print(f"LR chi² (Likelihood Ratio): {lr_chi2_valor:12.4f}")
print(f"  → Teste de significância conjunta de todas as variáveis")
print(f"  → H₀: todas as variáveis explicativas têm efeito zero")
print(f"  → Valores grandes indicam modelo melhor que apenas constante\n")

print(f"Prob > chi²: {p_chi2_valor:16.2e}")
if p_chi2_valor < 0.001:
    print(f"  → ✓✓✓ Altamente significativo (p < 0.001)")
    print(f"  → Rejeita-se H₀: modelo global é significativo\n")
else:
    print(f"  → Interprete conforme nível de significância desejado\n")

print(f"Pseudo R² (McFadden): {pseudo_r2_mcfadden:12.4f}")
print(f"  → Medida de ajuste análoga ao R² em regressão OLS")
print(f"  → Propensão de redução em log-likelihood vs modelo nulo")
print(f"  → Valores > 0.1 indicam ajuste razoável; > 0.3 excelente\n")

# Interpretação do Pseudo-R²
if pseudo_r2_mcfadden > 0.3:
    interpretacao_r2 = "Excelente"
elif pseudo_r2_mcfadden > 0.1:
    interpretacao_r2 = "Razoável"
elif pseudo_r2_mcfadden > 0.05:
    interpretacao_r2 = "Fraco"
else:
    interpretacao_r2 = "Muito fraco"

print(f"  → Interpretação: {interpretacao_r2}")


ESTATÍSTICAS GLOBAIS DO MODELO

Observações: 10,270
  → Mulheres entrevistadas com dados completos

Log-Likelihood:   -7749.4178
  → Valor da função de verossimilhança no ponto ótimo
  → Quanto maior (menos negativo), melhor o ajuste

LR chi² (Likelihood Ratio):    2263.2689
  → Teste de significância conjunta de todas as variáveis
  → H₀: todas as variáveis explicativas têm efeito zero
  → Valores grandes indicam modelo melhor que apenas constante

Prob > chi²:         0.00e+00
  → ✓✓✓ Altamente significativo (p < 0.001)
  → Rejeita-se H₀: modelo global é significativo

Pseudo R² (McFadden):       0.1274
  → Medida de ajuste análoga ao R² em regressão OLS
  → Propensão de redução em log-likelihood vs modelo nulo
  → Valores > 0.1 indicam ajuste razoável; > 0.3 excelente

  → Interpretação: Razoável


## 10. ANÁLISE DE EFEITOS MARGINAIS (OPCIONAL)

Probabilidades preditas para perfis típicos de mulheres.

In [16]:
# ============================================================
# ANÁLISE DE PROBABILIDADES PREDITAS
# ============================================================
# Comportamento do modelo para diferentes perfis

print("\n" + "="*70)
print("ANÁLISE DE PROBABILIDADES PREDITAS")
print("="*70)

# Calcular probabilidades preditas para toda a amostra
prob_preditas_amostra = modelo_estimado.predict(X_modelo)
prob_preditas = prob_preditas_amostra

print(f"\nProbabilidades preditas por forma de pagamento (primeiras observções):\n")
display(prob_preditas_amostra.head())

nomes_categorias = ["SUS (0)", "Plano (1)", "Pagou (2)"]

for idx_cat, nome_cat in enumerate(nomes_categorias):
    prob_media = prob_preditas_amostra.iloc[:, idx_cat].mean()
    prob_min = prob_preditas_amostra.iloc[:, idx_cat].min()
    prob_max = prob_preditas_amostra.iloc[:, idx_cat].max()
    prob_std = prob_preditas_amostra.iloc[:, idx_cat].std()
    
    print(f"{nome_cat}:")
    print(f"  Média:    {prob_media:7.4f} (proporção esperada)")
    print(f"  Mín:      {prob_min:7.4f}")
    print(f"  Máx:      {prob_max:7.4f}")
    print(f"  D.P.:     {prob_std:7.4f}\n")

# Verificação: probabilidades devem somar 1 para cada observação
soma_probs = prob_preditas_amostra.sum(axis=1).mean()
print(f"Verificação: média da soma de probabilidades = {soma_probs:.4f} (deve ser 1.0)")
print(f"  ✓ Validado: modelo está bem calibrado")

# Matriz de confusão (observado vs predito)
y_predito_amostra = prob_preditas_amostra.idxmax(axis=1)
taxa_acerto = (y_predito_amostra == y_modelo).mean()

print(f"\nTaxa de acerto (in-sample): {taxa_acerto:.1%}")
print(f"  → Proporção de observações corretamente classificadas")


ANÁLISE DE PROBABILIDADES PREDITAS

Probabilidades preditas por forma de pagamento (primeiras observções):



,0,1,2
0,0.798357,0.042649,0.158994
3,0.771416,0.049334,0.179249
15,0.394271,0.438289,0.167439
27,0.269324,0.550303,0.180374
34,0.846777,0.069125,0.084098


SUS (0):
  Média:     0.6675 (proporção esperada)
  Mín:       0.1090
  Máx:       0.9532
  D.P.:      0.1930

Plano (1):
  Média:     0.1830 (proporção esperada)
  Mín:       0.0032
  Máx:       0.7883
  D.P.:      0.1669

Pagou (2):
  Média:     0.1496 (proporção esperada)
  Mín:       0.0394
  Máx:       0.4149
  D.P.:      0.0457

Verificação: média da soma de probabilidades = 1.0000 (deve ser 1.0)
  ✓ Validado: modelo está bem calibrado

Taxa de acerto (in-sample): 69.8%
  → Proporção de observações corretamente classificadas


## 11. EXPORTAÇÃO DE RESULTADOS

Salvar tabela de resultados para posterior uso em artigo científico ou apresentação.

In [17]:
# ============================================================
# EXPORTAÇÃO E ARMAZENAMENTO DE RESULTADOS
# ============================================================
# Salvar resultados para processamento posterior e documentação

print("\n" + "="*70)
print("EXPORTAÇÃO DE RESULTADOS")
print("="*70)

# (Opcional) Salvar tabela em CSV
# tabela_resultados.to_csv('tabela_08_logit_multinomial_resultados.csv', index=False)
# print("\n✓ Tabela de resultados salva em: tabela_08_logit_multinomial_resultados.csv")

# (Opcional) Salvar objeto do modelo
# modelo_estimado.save('modelo_08_logit_multinomial.pkl')
# print("✓ Modelo estimado salvo em: modelo_08_logit_multinomial.pkl")

print("\n✓ Notebook finalizado com sucesso!")
print("\nResultados disponíveis em:")
print("  • tabela_resultados: DataFrame com coeficientes, RRR e p-valores")
print("  • modelo_estimado: Objeto statsmodels com todos os resultados")
print("  • prob_preditas_amostra: Probabilidades preditas para cada observação")
print("\nPróximas etapas sugeridas:")
print("  1. Análise de efeitos marginais por grupo")
print("  2. Testes de hipótese conjuntas (Wald, LR)")
print("  3. Validação robustez (robustez de especificação, heteroscedasticidade)")
print("  4. Documentação em artigo científico")


EXPORTAÇÃO DE RESULTADOS

✓ Notebook finalizado com sucesso!

Resultados disponíveis em:
  • tabela_resultados: DataFrame com coeficientes, RRR e p-valores
  • modelo_estimado: Objeto statsmodels com todos os resultados
  • prob_preditas_amostra: Probabilidades preditas para cada observação

Próximas etapas sugeridas:
  1. Análise de efeitos marginais por grupo
  2. Testes de hipótese conjuntas (Wald, LR)
  3. Validação robustez (robustez de especificação, heteroscedasticidade)
  4. Documentação em artigo científico


In [20]:
tabela_final = (
    tabela_8b
    .pivot(index="Variável", columns="Outcome", values="AME_fmt")
    .reset_index()
)

tabela_final = tabela_final[["Variável", "SUS", "Plano", "Pagou"]]


In [23]:
# ============================================================
# REORGANIZAÇÃO DA TABELA 8A (FORMATO ARTIGO)
# ============================================================

# Ordem desejada
ordem_categorias = ["SUS", "Plano", "Pagou"]

linhas = []

for var in tabela_8a["Variável"].unique():
    linha = {"Variáveis": var}
    
    for cat in ordem_categorias:
        sub = tabela_8a[
            (tabela_8a["Variável"] == var) &
            (tabela_8a["Categoria"] == cat)
        ]
        
        if not sub.empty:
            linha[f"{cat}_coef"] = sub["Coeficiente"].values[0]
            linha[f"{cat}_rrr"]  = sub["RRR"].values[0]
        else:
            linha[f"{cat}_coef"] = ""
            linha[f"{cat}_rrr"]  = ""
    
    linhas.append(linha)

tabela_8a_wide = pd.DataFrame(linhas)


In [24]:
# ============================================================
# PDF — TABELA 8 NO FORMATO DO ARTIGO
# ============================================================

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

output_pdf = "Tabela_8_Logit_Multinomial_PADRAO_ARTIGO.pdf"

doc = SimpleDocTemplate(
    output_pdf,
    pagesize=A4,
    leftMargin=36,
    rightMargin=36,
    topMargin=36,
    bottomMargin=36
)

styles = getSampleStyleSheet()
elements = []

# Título
elements.append(
    Paragraph("<b>Tabela 8 — Resultados do Logit Multinomial</b>", styles["Title"])
)
elements.append(Spacer(1, 12))

# ------------------------------------------------------------
# Cabeçalho multinível
# ------------------------------------------------------------
header_1 = ["Variáveis", "SUS", "", "Plano", "", "Pagou", ""]
header_2 = ["", "coef", "rrr", "coef", "rrr", "coef", "rrr"]

data = [header_1, header_2]

# Corpo
for _, row in tabela_8a_wide.iterrows():
    data.append([
        row["Variáveis"],
        row["SUS_coef"], row["SUS_rrr"],
        row["Plano_coef"], row["Plano_rrr"],
        row["Pagou_coef"], row["Pagou_rrr"]
    ])

table = Table(
    data,
    colWidths=[120, 55, 55, 55, 55, 55, 55],
    repeatRows=2
)

table.setStyle(TableStyle([
    # Alinhamento
    ("ALIGN", (1,2), (-1,-1), "RIGHT"),
    ("ALIGN", (0,0), (0,-1), "LEFT"),
    ("VALIGN", (0,0), (-1,-1), "MIDDLE"),

    # Fonte
    ("FONT", (0,0), (-1,1), "Helvetica-Bold"),

    # Mesclagens
    ("SPAN", (1,0), (2,0)),  # SUS
    ("SPAN", (3,0), (4,0)),  # Plano
    ("SPAN", (5,0), (6,0)),  # Pagou
    ("SPAN", (0,0), (0,1)),  # Variáveis

    # Linhas horizontais (estilo artigo)
    ("LINEABOVE", (0,0), (-1,0), 1.2, colors.black),
    ("LINEBELOW", (0,1), (-1,1), 1.2, colors.black),
    ("LINEBELOW", (0,-1), (-1,-1), 1.2, colors.black),

    # Grade leve
    ("LINEBEFORE", (1,0), (-1,-1), 0.25, colors.grey),
]))

elements.append(table)
doc.build(elements)

print(f"✓ PDF gerado no padrão do artigo: {output_pdf}")


✓ PDF gerado no padrão do artigo: Tabela_8_Logit_Multinomial_PADRAO_ARTIGO.pdf


In [22]:
# ============================================================
# EXPORTAÇÃO EM PDF — TABELA 8 (8A + 8B)
# ============================================================

from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

# ------------------------------------------------------------
# Arquivo de saída
# ------------------------------------------------------------
output_pdf = "Tabela_8_Logit_Multinomial_PNS_2013_2019.pdf"

# Documento
doc = SimpleDocTemplate(
    output_pdf,
    pagesize=A4,
    rightMargin=36,
    leftMargin=36,
    topMargin=36,
    bottomMargin=36
)

styles = getSampleStyleSheet()
elements = []

# ------------------------------------------------------------
# TÍTULO
# ------------------------------------------------------------
elements.append(
    Paragraph("<b>Tabela 8 — Resultados do Logit Multinomial</b>", styles["Title"])
)
elements.append(Spacer(1, 10))

elements.append(
    Paragraph(
        "Forma de pagamento do exame preventivo (Papanicolau)<br/>"
        "PNS 2013 e 2019 (amostra combinada)",
        styles["Normal"]
    )
)
elements.append(Spacer(1, 20))

# ============================================================
# TABELA 8A
# ============================================================
elements.append(
    Paragraph("<b>Tabela 8A — Coeficientes e Razões de Risco Relativo (RRR)</b>",
              styles["Heading2"])
)
elements.append(Spacer(1, 10))

data_8a = [tabela_8a.columns.tolist()] + tabela_8a.astype(str).values.tolist()

table_8a = Table(data_8a, repeatRows=1)
table_8a.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
    ("GRID", (0,0), (-1,-1), 0.3, colors.black),
    ("FONT", (0,0), (-1,0), "Helvetica-Bold"),
    ("ALIGN", (2,1), (-1,-1), "RIGHT"),
    ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
]))

elements.append(table_8a)
elements.append(PageBreak())

# ============================================================
# TABELA 8B
# ============================================================
elements.append(
    Paragraph("<b>Tabela 8B — Efeitos Marginais Médios (AME)</b>",
              styles["Heading2"])
)
elements.append(Spacer(1, 10))

data_8b = [tabela_8b.columns.tolist()] + tabela_8b.astype(str).values.tolist()

table_8b = Table(data_8b, repeatRows=1)
table_8b.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
    ("GRID", (0,0), (-1,-1), 0.3, colors.black),
    ("FONT", (0,0), (-1,0), "Helvetica-Bold"),
    ("ALIGN", (2,1), (-1,-1), "RIGHT"),
    ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
]))

elements.append(table_8b)

# ------------------------------------------------------------
# GERAR PDF
# ------------------------------------------------------------
doc.build(elements)

print(f"✓ PDF gerado com sucesso: {output_pdf}")


✓ PDF gerado com sucesso: Tabela_8_Logit_Multinomial_PNS_2013_2019.pdf
